# 5.4 poskyris. Rezultatai su sumažintu LIEPA-1 + DEMAND duomenų rinkiniu

## 1. Importai ir nustatymai

In [ ]:
# 1. Importai ir nustatymai
import pandas as pd
import numpy as np
import json
import os
import csv
import warnings
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        9,
    'axes.titlesize':   11,
    'axes.labelsize':   10,
    'xtick.labelsize':  9,
    'ytick.labelsize':  9,
    'figure.dpi':       150,
})

CSV_PATH    = 'results/all_runs.csv'
RESULTS_DIR = 'results'
FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

EXP_TARGET  = 'full_liepa_unetdilatedmaskcnn_v2'
EXP_SKIP    = 'full_liepa_full_unetdilatedmaskcnn_v2'   # pilnas rinkinys – nenaudoti

def lt_num(v, dec=3):
    """Skaičius su lietuvišku dešimtainiu kableliu."""
    return f'{v:.{dec}f}'.replace('.', ',')

print("Nustatymai įkelti.")
print(f"Tikslinis eksperimentas : {EXP_TARGET}")
print(f"Ignoruojamas eksperimentas: {EXP_SKIP}")


## 2. Duomenų nuskaitymas ir diagnostika

In [ ]:
# 2. Duomenų nuskaitymas ir diagnostika
df = pd.read_csv(CSV_PATH)

print("Stulpeliai:")
print(list(df.columns))
print(f"\nIš viso eilučių: {len(df)}")
print()

print("=== Pirmos 3 eilutes ===")
display(df.head(3))
print()

# Visi full_liepa eksperimentai
fl_mask = df['experiment_id'].str.contains('full_liepa', case=False, na=False)
print("=== Visi full_liepa eksperimentai ===")
display(df[fl_mask][[
    'experiment_id', 'dataset_name', 'model_name',
    'loss_name', 'learning_rate', 'n_fft', 'segment_seconds',
    'num_eval_files', 'best_val_loss', 'best_epoch', 'epochs_completed'
]])
print()
print(f"Pastaba: '{EXP_SKIP}' nenaudojamas – tai pilnas LIEPA-1 + DEMAND rinkinys,")
print("         skirtas vėlesniam poskyriui.")


## 3. Sumažinto LIEPA eksperimento radimas

In [ ]:
# 3. Sumažinto LIEPA eksperimento radimas
rows = df[df['experiment_id'] == EXP_TARGET]

if len(rows) == 0:
    # Parodyti artimiausius kandidatus
    candidates = df[
        df['experiment_id'].str.contains('full_liepa', case=False, na=False) &
        ~df['experiment_id'].str.contains('full_', regex=False)
    ]
    print(f"ISPEJIMAS: '{EXP_TARGET}' nerastas!")
    print("Artimiausi kandidatai:")
    display(candidates[['experiment_id','model_name','loss_name',
                         'learning_rate','num_eval_files']])
    raise RuntimeError(f"Eksperimentas '{EXP_TARGET}' nerastas all_runs.csv!")

if len(rows) > 1:
    print(f"Rasta {len(rows)} eiluciu. Naudojama paskutine.")
    row = rows.iloc[-1]
else:
    row = rows.iloc[0]

print(f"Rastas: {row['experiment_id']}")
print(f"  Modelis:          {row['model_name']}")
print(f"  Duomenu rinkinys: {row['dataset_name']}")
print(f"  Nuostoliu f-ja:   {row['loss_name']}")
print(f"  Mokymosi greitis: {row['learning_rate']}")
print(f"  STFT langas:      {row['n_fft']}")
print(f"  Segmentas, s:     {row['segment_seconds']}")
print(f"  Testavimo failai: {row['num_eval_files']}")


## 4. Reikšmių ištraukimas ir patikra

In [ ]:
# 4. Reikšmių ištraukimas ir patikra

# --- iš all_runs.csv ---
pesq_before  = float(row['mean_pesq_noisy'])
pesq_after   = float(row['mean_pesq_enhanced'])
delta_pesq   = float(row['mean_delta_pesq'])
stoi_before  = float(row['mean_stoi_noisy'])
stoi_after   = float(row['mean_stoi_enhanced'])
delta_stoi   = float(row['mean_delta_stoi'])
delta_snr    = float(row['mean_delta_snr_est'])
snr_before   = float(row['mean_snr_noisy_est'])
snr_after    = float(row['mean_snr_enhanced_est'])
n_files      = int(row['num_eval_files'])
best_val_csv = float(row['best_val_loss'])
best_epoch   = int(row['best_epoch'])
epochs_done  = int(row['epochs_completed'])
max_epochs   = int(row['epochs'])
n_fft        = int(row['n_fft'])
seg_s        = float(row['segment_seconds'])
sr           = int(row['target_sample_rate'])
loss_name    = str(row['loss_name'])
lr           = float(row['learning_rate'])

n_skipped = int(row['num_skipped_files'])     if 'num_skipped_files' in row and not pd.isna(row['num_skipped_files'])     else None

n_params = int(row['num_parameters'])     if 'num_parameters' in row and not pd.isna(row['num_parameters'])     else None

# --- papildoma informacija iš run_summary.json ---
json_path = os.path.join(RESULTS_DIR, EXP_TARGET, 'run_summary.json')
if os.path.exists(json_path):
    with open(json_path, encoding='utf-8') as f:
        summary = json.load(f)
    patience     = summary.get('patience', 'N/A')
    min_delta    = summary.get('min_delta', 'N/A')
    early_stop   = summary.get('early_stopping', 'N/A')
    use_sched    = summary.get('use_lr_scheduler', 'N/A')
    sched_pat    = summary.get('lr_scheduler_patience', 'N/A')
    sched_fac    = summary.get('lr_scheduler_factor', 'N/A')
    if n_params is None:
        n_params = summary.get('num_parameters', None)
    if n_skipped is None:
        n_skipped = summary.get('num_skipped_files', 0)
    best_val     = float(summary['best_val_loss'])
    print(f"run_summary.json rastas: {json_path}")
else:
    print(f"ISPEJIMAS: {json_path} nerastas – naudojami tik CSV duomenys.")
    patience = early_stop = use_sched = sched_pat = sched_fac = 'N/A'
    min_delta = 'N/A'
    best_val  = best_val_csv
    summary   = {}

print()
print("=== Isrinktos reiksmes ===")
print(f"  PESQ pries apdorojima:   {lt_num(pesq_before)}")
print(f"  PESQ po apdorojimo:      {lt_num(pesq_after)}")
print(f"  DPESQ:                   {lt_num(delta_pesq)}")
print(f"  STOI pries apdorojima:   {lt_num(stoi_before)}")
print(f"  STOI po apdorojimo:      {lt_num(stoi_after)}")
print(f"  DSTOI:                   {lt_num(delta_stoi)}")
print(f"  DSNR, dB:                {lt_num(delta_snr, 2)}")
print(f"  Testavimo failai:        {n_files}")
print(f"  Praleisti failai:        {n_skipped if n_skipped is not None else 'N/A'}")
print(f"  Geriausias val. nuost.:  {lt_num(best_val, 6)}")
print(f"  Geriausia epocha:        {best_epoch}")
print(f"  Mokyta epochu:           {epochs_done} / {max_epochs}")
print(f"  Modelio parametrai:      {n_params:,}" if n_params else "  Modelio parametrai:      N/A")
print(f"  n_fft:                   {n_fft}")
print(f"  Segmento trukme, s:      {seg_s}")
print(f"  Diskretizavimo daznis:   {sr} Hz")
print(f"  Nuostoliu funkcija:      {loss_name}")
print(f"  Mokymosi greitis:        {lr}")


## 5. history.csv nuskaitymas

In [ ]:
# 5. history.csv nuskaitymas
hist_path = os.path.join(RESULTS_DIR, EXP_TARGET, 'history.csv')
if not os.path.exists(hist_path):
    raise FileNotFoundError(f"Nerastas: {hist_path}")

hist = pd.read_csv(hist_path)
print("history.csv stulpeliai:", list(hist.columns))
print(f"Eiluciu (epochu): {len(hist)}")
display(hist.head(5))

# Geriausia epocha iš improved žymos
if 'improved' in hist.columns:
    improved_epochs = hist[hist['improved'] == True]['epoch'].tolist()
    _best_ep_hist   = int(hist[hist['improved'] == True]['epoch'].max())
    print(f"\nimproved=True epochos: {improved_epochs}")
    print(f"Paskutine improved epocha: {_best_ep_hist}")
else:
    _best_ep_hist = int(hist.loc[hist['val_loss'].idxmin(), 'epoch'])
    print(f"ISPEJIMAS: 'improved' stulpelio nera. Geriausia epocha pagal min val_loss: {_best_ep_hist}")

print(f"Galutinis mokymosi greitis: {hist.iloc[-1]['learning_rate']}")
_final_lr = float(hist.iloc[-1]['learning_rate'])


## 6. Nuostolių kreivė


In [ ]:
# 6. Loss curve generavimas – 26 pav.
_ep_vals  = hist['epoch'].values
_tr_vals  = hist['train_loss'].values
_vl_vals  = hist['val_loss'].values
_best_ep  = best_epoch     # iš run_summary.json
_best_vl  = float(hist[hist['epoch'] == _best_ep]['val_loss'].iloc[0])
_ep_done  = int(hist['epoch'].max())

fig, ax = plt.subplots(figsize=(7.5, 4.0))

ax.plot(_ep_vals, _tr_vals, color='#2171B5', linewidth=1.4,
        label='Mokymo nuostolis')
ax.plot(_ep_vals, _vl_vals, color='#EF6C00', linewidth=1.4,
        linestyle='--', label='Validavimo nuostolis')
ax.axvline(x=_best_ep, color='#388E3C', linewidth=1.2, linestyle=':',
           label=f'Geriausia epocha ({_best_ep})')
ax.scatter([_best_ep], [_best_vl], color='#388E3C', zorder=5, s=42)

ax.set_xlabel('Epocha', fontsize=10)
ax.set_ylabel('Nuostolis', fontsize=10)
ax.set_title(
    'Sumažinto LIEPA-1 + DEMAND rinkinio mokymo ir\n'
    'validavimo nuostolių kaita',
    fontsize=10, fontweight='bold', pad=8
)

ax.set_xlim(0, _ep_done + 1)
_ymax = max(float(_tr_vals.max()), float(_vl_vals.max())) * 1.08
ax.set_ylim(0, _ymax)
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda v, _: f'{v:.4f}'.replace('.', ','))
)
ax.grid(linestyle='--', linewidth=0.5, alpha=0.55)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper right', fontsize=8.5, framealpha=0.85)

_bvl_str = lt_num(_best_vl, 6)
ax.annotate(
    f'Ankstyvasis stabdymas: {_ep_done} / {max_epochs} epochu. '
    f'Geriausia val. nuost.: {_bvl_str} ({_best_ep} ep.)',
    xy=(0.01, 0.02), xycoords='axes fraction',
    fontsize=7.5, color='#555555', va='bottom', ha='left'
)

plt.tight_layout()

_png26 = os.path.join(FIGURES_DIR, '26_liepa_matched_loss_curve.png')
_svg26 = os.path.join(FIGURES_DIR, '26_liepa_matched_loss_curve.svg')
fig.savefig(_png26, dpi=300, bbox_inches='tight')
fig.savefig(_svg26, bbox_inches='tight')
plt.show()
print(f"Issaugota: {_png26}")
print(f"Issaugota: {_svg26}")


## 7. PESQ ir STOI rezultatai

In [ ]:
# 7. PESQ / STOI stulpelinė diagrama – 27 pav.
LABELS = ['Prieš\napdorojimą', 'Po\napdorojimo']
BAR_W  = 0.45
COLORS = ['#5B9BD5', '#70AD47']
EDGE   = '#333333'

# y-ribos: paskaiciuoti dinamiškai su nedidele paraštė
def _ylims(a, b, margin=0.12):
    rng = b - a
    lo  = max(0.0, a - rng * margin)
    hi  = b + rng * margin
    return lo, hi

pesq_lo, pesq_hi = _ylims(pesq_before, pesq_after)
stoi_lo, stoi_hi = _ylims(stoi_before, stoi_after)

# Jeigu STOI riba per uzkerta 0, nustatyti min 0.85
stoi_lo = max(stoi_lo, 0.85)

fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.8))

for ax, subtitle, vals, ylim_lo, ylim_hi in [
    (axes[0], 'PESQ', [pesq_before, pesq_after], pesq_lo, pesq_hi),
    (axes[1], 'STOI', [stoi_before, stoi_after], stoi_lo, stoi_hi),
]:
    x    = np.arange(len(vals))
    bars = ax.bar(x, vals, width=BAR_W, color=COLORS,
                  edgecolor=EDGE, linewidth=0.8)
    pad = (ylim_hi - ylim_lo) * 0.015
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + pad,
            lt_num(v, 3),
            ha='center', va='bottom', fontsize=9, fontweight='bold'
        )
    ax.set_xticks(x)
    ax.set_xticklabels(LABELS, fontsize=9)
    ax.set_ylabel(subtitle, fontsize=10)
    ax.set_ylim(ylim_lo, ylim_hi)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, _: lt_num(v, 2))
    )
    ax.set_title(subtitle, fontsize=11, fontweight='bold', pad=6)
    ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.6)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(
    'Sumažinto LIEPA-1 + DEMAND rinkinio PESQ ir STOI rezultatai\n'
    'prieš ir po apdorojimo',
    fontsize=10, y=1.02
)
plt.tight_layout()

_png27 = os.path.join(FIGURES_DIR, '27_liepa_matched_pesq_stoi_pries_po.png')
_svg27 = os.path.join(FIGURES_DIR, '27_liepa_matched_pesq_stoi_pries_po.svg')
fig.savefig(_png27, dpi=300, bbox_inches='tight')
fig.savefig(_svg27, bbox_inches='tight')
plt.show()
print(f"Issaugota: {_png27}")
print(f"Issaugota: {_svg27}")


## 8. CSV lentelių išsaugojimas

In [ ]:
# 8. CSV lentelių išsaugojimas

# --- 8a. Rezultatai ---
csv1 = os.path.join(FIGURES_DIR, 'liepa_matched_rezultatai.csv')
with open(csv1, 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f)
    w.writerow(['Rodiklis', 'Pries apdorojima', 'Po apdorojimo', 'Pokytis'])
    w.writerow(['PESQ',     lt_num(pesq_before,  3), lt_num(pesq_after,  3), lt_num(delta_pesq, 3)])
    w.writerow(['STOI',     lt_num(stoi_before,  3), lt_num(stoi_after,  3), lt_num(delta_stoi, 3)])
    w.writerow(['DSNR, dB', '-',                     '-',                    lt_num(delta_snr,  2)])
print(f"Issaugota: {csv1}")

# --- 8b. Eksperimento konfigūracijos santrauka ---
csv2 = os.path.join(FIGURES_DIR, 'liepa_matched_eksperimento_santrauka.csv')
ankst = 'Taip' if str(early_stop).lower() in ('true', '1') else str(early_stop)
n_par_str = f'{n_params:,}' if n_params else 'N/A'
rows2 = [
    ('Eksperimentas',             EXP_TARGET),
    ('Duomenu rinkinys',          'LIEPA-1 + DEMAND (sumazintas)'),
    ('Modelis',                   row['model_name']),
    ('Nuostoliu funkcija',        loss_name),
    ('Mokymosi greitis',          str(lr)),
    ('STFT lango dydis',          str(n_fft)),
    ('Segmento trukme',           f'{int(seg_s)} s'),
    ('Diskretizavimo daznis',     f'{sr} Hz'),
    ('Optimizavimo algoritmas',   'Adam'),
    ('Modelio parametru skaicius', n_par_str),
    ('Maksimalus epochu skaicius', str(max_epochs)),
    ('Faktiskai mokyta epochu',   str(epochs_done)),
    ('Geriausia epocha',          str(best_epoch)),
    ('Geriausias validavimo nuostolis', lt_num(best_val, 6)),
    ('Ankstyvasis stabdymas',     ankst),
    ('Testavimo failu skaicius',  str(n_files)),
    ('Pralestu failu skaicius',   str(n_skipped if n_skipped is not None else 0)),
]
with open(csv2, 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f)
    w.writerow(['Parametras', 'Reiksme'])
    w.writerows(rows2)
print(f"Issaugota: {csv2}")

print()
display(pd.read_csv(csv1, encoding='utf-8-sig'))
print()
display(pd.read_csv(csv2, encoding='utf-8-sig'))
